In [ ]:
import argparse
import json
import concurrent.futures
import re
import os
import sys
from tqdm import tqdm
from PIL import Image
from openai import OpenAI
import multiprocessing
import random
import base64
from pathlib import Path


def fill_image_captions_in_md(md_path, json_path, middle_json_path, output_md_path):
    """
    将 md 文件中 ![](...) 的 alt-text 填充为归一化的 bbox 信息。
    bbox 来自 json_path，图片尺寸来自 image_paths。
    """
    with open(md_path, "r", encoding="utf-8") as f:
        md_content = f.read()

    with open(json_path, "r", encoding="utf-8") as f:
        content_list = json.load(f)
    
    with open(middle_json_path, "r", encoding="utf-8") as f:
        middle_content_list = json.load(f)

    # --- 步骤 1: 创建 page_idx -> image_size 的映射 ---
    page_to_size = {}
    for page_res in middle_content_list["pdf_info"]:
        page_idx = page_res["page_idx"]
        page_size = page_res.get("page_size")
        if page_idx in page_to_size:
            print(f"[WARN] 已经找到 page_idx {page_idx} 对应的图片尺寸，跳过此页的 bbox 映射。")
        else:
            page_to_size[page_idx] = page_size

    # --- 步骤 2: 构建 img_path -> 页面和BBox信息的映射 ---
    img_map = {}
    for page_res in middle_content_list["pdf_info"]:
        page_idx = page_res["page_idx"]
        images = page_res.get("images", None)
        if not images:  # 该页中没有图片
            continue

        for image in images:
            for block in image["blocks"]:
                for line in block["lines"]:
                    for item in line["spans"]:
                        # print(item)
                        if item.get("type") == "image":
                            img_path = item.get("image_path")    
                            bbox = item.get("bbox")
                            
                            if img_path and isinstance(page_idx, int) and bbox and len(bbox) == 4:
                                img_map[f"images/{img_path}"] = {
                                    "page_idx": page_idx,
                                    "bbox": bbox,
                                }
                        else:
                            print(f"[WARN] {middle_json_path} JSON中图片条目信息不完整，已跳过: {item}")

    # --- 步骤 3: 修改替换逻辑 ---
    def replace_image(match):
        img_path = match.group(2)  # () 中的内容

        if img_path in img_map:
            info = img_map[img_path]
            page_idx = info["page_idx"]
            x0, y0, x1, y1 = info["bbox"]

            if page_idx in page_to_size:
                width, height = page_to_size[page_idx]
                
                if width == 0 or height == 0:
                    print(f"[WARN] ❌ {md_path} 图片 {img_path} (page {page_idx}) 的尺寸为零，无法归一化。")
                    return match.group(0)

                # 计算归一化坐标 (0-1000)
                norm_x0 = round((x0 / width) * 1000)
                norm_y0 = round((y0 / height) * 1000)
                norm_x1 = round((x1 / width) * 1000)
                norm_y1 = round((y1 / height) * 1000)

                final_filename = f"page{page_idx+1}_{int(norm_x0)}_{int(norm_y0)}_{int(norm_x1)}_{int(norm_y1)}.jpg"
                alt_text = ""
                return f"![{alt_text}]({final_filename})"
            else:
                print(f"[WARN] ❌ {md_path} 匹配到 img_path {img_path}, 但 page_idx {page_idx} 匹配不上。")
        else:
            print(f"[WARN] ❌ {md_path} 未找到匹配: img_path={img_path}")      
        return match.group(0)

    new_content = re.sub(r'!\[([^\]]*)\]\(([^)]+)\)', replace_image, md_content)

    output_md_path = Path(output_md_path)
    output_md_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_md_path, "w", encoding="utf-8") as f:
        f.write(new_content)

INPUT_FILE = "MPDocBench.json"
OUTPUT_PATH = "./markdown/monkeyocrpro_3b"

MD_PATH = OUTPUT_PATH + "_md"
os.makedirs(MD_PATH, exist_ok=True)
MD_PATH = Path(MD_PATH)

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)
    raw_data = {}
    for item in data:
        page_info = item["page_info"]
        images_list = page_info["images_list"]
        annotations_list = page_info["annotations_list"]
        image_path = page_info["image_path"]
        pdf_name = os.path.splitext(image_path)[0]
        if pdf_name not in raw_data:
            raw_data[pdf_name] = []
            page_id = 0
            for img, ann in zip(images_list, annotations_list):
                raw_data[pdf_name].append((page_id, img, ann))
                page_id += 1
        else:
            print(f"Warning: duplicate pdf_name {pdf_name} found. Skipping.")
            continue
    data = raw_data

for pdf_name in data:
    pdf_dir = Path(os.path.join(OUTPUT_PATH, pdf_name))
    md_name = ""
    for file in os.listdir(pdf_dir):
        if file.endswith(".md"):
            md_name = os.path.splitext(file)[0]
            break
    
    md_path = pdf_dir / f"{md_name}.md"
    json_path = pdf_dir / f"{md_name}_content_list.json"
    middle_json_path = pdf_dir / f"{md_name}_middle.json"
    md_final_path = MD_PATH / f"{pdf_name}.md"

    fill_image_captions_in_md(md_path, json_path, middle_json_path, output_md_path=md_final_path)